# Lesson 19 Lab — KV-Cache Quantization for Long Contexts

**Puzzle:** When context length doubles, why can KV cache dominate even after weight quantization?

The saved outputs were generated by executing every code cell on the recorded RTX 5090. Run all cells to regenerate the evidence on your own CUDA GPU.

## 0. Predict before running

Write down: (1) the expected direction, (2) the mechanism, (3) the observation that would reverse your prediction, and (4) the evidence level required for the claim.

## 1. Theory — objects and data flow

The KV cache stores keys and values per layer and request. Quantized cache additionally stores scales (and sometimes zero points) at a chosen token/head/block granularity.

### Core mechanism

Cache bytes follow `2LBTHD·bytes`, while attention uses `softmax(QKᵀ/√D)V`; quantization error can perturb both logits through `K` and the weighted sum through `V`.

In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "19-kv-cache-quantization"
device = require_cuda()
torch.manual_seed(2026 + 19)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 2. Connect theory to the experiment

### Engineering trade-off

Fine-grained scales reduce error but add metadata and Q/DQ work. Capacity gains can improve concurrency even if a single request pays extra latency.

### What this code tests

The notebook quantizes real CUDA K/V tensors, includes scale bytes, and compares attention outputs rather than reporting compression alone.

**Experiment:** Quantize representative KV tensors to INT8 on CUDA, compare bytes and attention-output error, and project capacity across context lengths.

**Declared evidence label:** `pytorch-gpu`. Check that the shapes, controlled variables, and units match the theoretical question before executing.

In [2]:
batch,heads,seq,dim=1,8,4096,128; k=torch.randn(batch,heads,seq,dim,device=device); v=torch.randn_like(k); q=torch.randn(batch,heads,1,dim,device=device)
def qdq(t):
    scale=t.abs().amax(-1,keepdim=True).clamp_min(1e-8)/127; qt=torch.round(t/scale).clamp(-128,127).to(torch.int8); return qt,scale,qt.float()*scale
qk,sk,kd=qdq(k); qv,sv,vd=qdq(v)
ref=torch.softmax(q@k.transpose(-1,-2)/(dim**0.5),-1)@v; cand=torch.softmax(q@kd.transpose(-1,-2)/(dim**0.5),-1)@vd
bf16_bytes=2*(k.numel()+v.numel()); int8_bytes=qk.numel()+qv.numel()+sk.numel()*sk.element_size()+sv.numel()*sv.element_size()
result=base_result(19,"pytorch-gpu"); result.update({"shape":{"batch":batch,"heads":heads,"sequence":seq,"head_dim":dim},
    "bf16_bytes":bf16_bytes,"int8_plus_scale_bytes":int8_bytes,"memory_reduction_pct":round((1-int8_bytes/bf16_bytes)*100,4),
    "attention_output_error":error_metrics(ref,cand),"conclusion":"INT8 cache reduced storage in this reference while introducing measurable attention-output error."})


## 3. Inspect the evidence

Report cache bytes, metadata, attention error, and any quantize/dequantize overhead separately.

### Acceptance and rollback gate

Measure actual cache allocation, metadata, context-dependent attention or task error, quant/dequant cost, long-context quality, and end-to-end serving metrics.

In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "attention_output_error": {
    "cosine": 0.99995804,
    "mae": 0.00018211,
    "max_abs": 0.00070267,
    "rmse": 0.00023131
  },
  "bf16_bytes": 16777216,
  "conclusion": "INT8 cache reduced storage in this reference while introducing measurable attention-output error.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:45:58+00:00",
  "int8_plus_scale_bytes": 8650752,
  "lesson": 19,
  "memory_reduction_pct": 48.4375,
  "schema_version": 1,
  "shape": {
    "batch": 1,
    "head_dim": 128,
    "heads": 8,
    "sequence": 4096
  }
}
Saved: artifacts/rtx5090-result.json


## 4. Explain the result

KV quantization is primarily a capacity decision until end-to-end latency and quality are measured.

Relate the measured fields back to the mechanism above. Treat the checked-in result as one hardware/software observation, not a universal ranking. The complete derivation, evidence boundary, and primary references are in [`README.md`](README.md).